# 02 - Preprocessing V2 (True Forecasting, t+1)

This version creates a true future target:
- `AQI_t_plus_1 = AQI.shift(-1)` per city
- feature row at time `t`
- target at time `t+1`

Leakage controls:
- no backward fill
- train/test split by **target timestamp**
- outlier caps and scaler fit on train only
- no fitting on test data


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

INPUT_PATH = Path('data/processed/combined_all_cities.csv')
SPLITS_DIR = Path('data/splits')
MODELS_DIR = Path('outputs/models')

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TARGET_START = pd.Timestamp('2020-01-01')
TRAIN_TARGET_END = pd.Timestamp('2023-12-31')
TEST_TARGET_START = pd.Timestamp('2024-01-01')
TEST_TARGET_END = pd.Timestamp('2024-12-31')

REQUIRED_COLUMNS = ['Timestamp', 'City', 'Location', 'PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3', 'AQI']
POLLUTANT_COLUMNS = ['PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3']
FEATURE_COLUMNS = [
    'PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3',
    'PM2.5_lag1', 'PM2.5_lag7', 'PM2.5_lag30',
    'AQI_lag1', 'AQI_lag7',
    'PM2.5_roll7_mean', 'PM2.5_roll7_std', 'PM2.5_roll30_mean',
    'day_of_week', 'month', 'season', 'is_weekend', 'year'
]
LAG_ROLL_REQUIRED = [
    'PM2.5_lag1', 'PM2.5_lag7', 'PM2.5_lag30',
    'AQI_lag1', 'AQI_lag7',
    'PM2.5_roll7_mean', 'PM2.5_roll7_std', 'PM2.5_roll30_mean'
]


## Load, Validate, Basic Cleaning

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Missing input: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH)

if 'Month' in df.columns:
    df = df.drop(columns=['Month'])
    print('Dropped existing Month column.')

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
bad_ts = int(df['Timestamp'].isna().sum())
if bad_ts:
    print(f'Dropping invalid Timestamp rows: {bad_ts}')
df = df.dropna(subset=['Timestamp']).copy()

for col in POLLUTANT_COLUMNS + ['AQI']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values(['City', 'Timestamp']).reset_index(drop=True)
print('Base shape:', df.shape)
print('Base date range:', df['Timestamp'].min().date(), 'to', df['Timestamp'].max().date())


## Missing Handling (Causal)

Use only forward fill (`ffill(limit=3)`) per city. No backward fill to avoid future leakage.

In [ ]:
before_null = df[POLLUTANT_COLUMNS + ['AQI']].isna().sum().to_dict()
grp = df.groupby('City', group_keys=False)
df[POLLUTANT_COLUMNS] = grp[POLLUTANT_COLUMNS].ffill(limit=3)
df['AQI'] = grp['AQI'].ffill(limit=3)

df = df.dropna(subset=['PM2.5', 'AQI']).copy()
after_null = df[POLLUTANT_COLUMNS + ['AQI']].isna().sum().to_dict()
print('Nulls before:', before_null)
print('Nulls after :', after_null)
print('Rows after null drop:', len(df))


## Feature Engineering At Time t + Future Target At t+1

In [ ]:
grp = df.groupby('City', group_keys=False)

# Future target and its timestamp
df['AQI_t_plus_1'] = grp['AQI'].shift(-1)
df['target_timestamp'] = grp['Timestamp'].shift(-1)

# Causal lag features (strictly past)
df['PM2.5_lag1'] = grp['PM2.5'].shift(1)
df['PM2.5_lag7'] = grp['PM2.5'].shift(7)
df['PM2.5_lag30'] = grp['PM2.5'].shift(30)
df['AQI_lag1'] = grp['AQI'].shift(1)
df['AQI_lag7'] = grp['AQI'].shift(7)

# Causal rolling features at time t (window uses data <= t only)
df['PM2.5_roll7_mean'] = grp['PM2.5'].transform(lambda s: s.rolling(window=7, min_periods=7).mean())
df['PM2.5_roll7_std'] = grp['PM2.5'].transform(lambda s: s.rolling(window=7, min_periods=7).std())
df['PM2.5_roll30_mean'] = grp['PM2.5'].transform(lambda s: s.rolling(window=30, min_periods=30).mean())

df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['month'] = df['Timestamp'].dt.month
df['year'] = df['Timestamp'].dt.year
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

def month_to_season(m):
    if m in (12, 1, 2):
        return 0
    if m in (3, 4, 5):
        return 1
    if m in (6, 7, 8):
        return 2
    return 3

df['season'] = df['month'].apply(month_to_season).astype(int)

before_drop = len(df)
df = df.dropna(subset=LAG_ROLL_REQUIRED + ['AQI_t_plus_1', 'target_timestamp']).copy()
print('Dropped due to lag/rolling/future-target nulls:', before_drop - len(df))
print('Rows after feature+target build:', len(df))


## Split By Target Timestamp (Forecast Horizon Integrity)

In [ ]:
train_mask = (df['target_timestamp'] >= TRAIN_TARGET_START) & (df['target_timestamp'] <= TRAIN_TARGET_END)
test_mask = (df['target_timestamp'] >= TEST_TARGET_START) & (df['target_timestamp'] <= TEST_TARGET_END)

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

assert not train_df.empty, 'Train split is empty.'
assert not test_df.empty, 'Test split is empty.'
assert train_df['target_timestamp'].max() < test_df['target_timestamp'].min(), 'Target-date overlap between train and test.'
assert (df['Timestamp'] < df['target_timestamp']).all(), 'Non-causal rows detected: feature time must be < target time.'

print('Train rows:', len(train_df))
print('Test rows :', len(test_df))
print('Train feature time range:', train_df['Timestamp'].min().date(), 'to', train_df['Timestamp'].max().date())
print('Train target  time range:', train_df['target_timestamp'].min().date(), 'to', train_df['target_timestamp'].max().date())
print('Test feature time range :', test_df['Timestamp'].min().date(), 'to', test_df['Timestamp'].max().date())
print('Test target  time range :', test_df['target_timestamp'].min().date(), 'to', test_df['target_timestamp'].max().date())


## Fit Outlier Caps On Train Only, Apply To Both

In [ ]:
bounds = {}
for city, grp_city in train_df.groupby('City'):
    bounds[city] = {}
    for col in POLLUTANT_COLUMNS:
        s = grp_city[col].dropna()
        if s.empty:
            bounds[city][col] = np.inf
            continue
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        bounds[city][col] = q3 + 3.0 * iqr

def apply_caps(split_df, caps):
    out = split_df.copy()
    cap_counts = {c: 0 for c in POLLUTANT_COLUMNS}
    for city, idx in out.groupby('City').groups.items():
        for col in POLLUTANT_COLUMNS:
            upper = caps.get(city, {}).get(col, np.inf)
            if np.isfinite(upper):
                orig = out.loc[idx, col]
                cap_counts[col] += int((orig > upper).sum())
                out.loc[idx, col] = orig.clip(upper=upper)
    return out, cap_counts

train_df, train_cap_counts = apply_caps(train_df, bounds)
test_df, test_cap_counts = apply_caps(test_df, bounds)

print('Capped counts (train):', train_cap_counts)
print('Capped counts (test) :', test_cap_counts)


## Scale Features (Fit On Train Only) And Save Artifacts

In [ ]:
missing_feat = [c for c in FEATURE_COLUMNS if c not in train_df.columns]
if missing_feat:
    raise ValueError(f'Missing feature columns: {missing_feat}')

scaler = StandardScaler()

train_scaled = pd.DataFrame(
    scaler.fit_transform(train_df[FEATURE_COLUMNS].astype(float)),
    columns=FEATURE_COLUMNS,
    index=train_df.index,
)
test_scaled = pd.DataFrame(
    scaler.transform(test_df[FEATURE_COLUMNS].astype(float)),
    columns=FEATURE_COLUMNS,
    index=test_df.index,
)

for col in FEATURE_COLUMNS:
    train_df[col] = train_scaled[col]
    test_df[col] = test_scaled[col]

joblib.dump(scaler, MODELS_DIR / 'scaler_v2.pkl')

# Save both versioned and default feature schema for downstream compatibility.
(MODELS_DIR / 'feature_columns_v2.json').write_text(json.dumps(FEATURE_COLUMNS, indent=2), encoding='utf-8')
(MODELS_DIR / 'feature_columns.json').write_text(json.dumps(FEATURE_COLUMNS, indent=2), encoding='utf-8')

print('Saved:', MODELS_DIR / 'scaler_v2.pkl')
print('Saved:', MODELS_DIR / 'feature_columns_v2.json')
print('Updated:', MODELS_DIR / 'feature_columns.json')


## Final Leakage Validation + Export V2 Splits

In [ ]:
# Leakage validation: no feature timestamps at/after target, no missing target.
assert (train_df['Timestamp'] < train_df['target_timestamp']).all()
assert (test_df['Timestamp'] < test_df['target_timestamp']).all()
assert train_df['AQI_t_plus_1'].isna().sum() == 0
assert test_df['AQI_t_plus_1'].isna().sum() == 0

export_columns = ['Timestamp', 'target_timestamp', 'City', 'Location', 'AQI_t_plus_1'] + FEATURE_COLUMNS
train_out = train_df[export_columns].copy().rename(columns={'AQI_t_plus_1': 'AQI'})
test_out = test_df[export_columns].copy().rename(columns={'AQI_t_plus_1': 'AQI'})

train_out.to_csv(SPLITS_DIR / 'train_v2.csv', index=False)
test_out.to_csv(SPLITS_DIR / 'test_v2.csv', index=False)

print('Saved:', SPLITS_DIR / 'train_v2.csv', '| rows =', len(train_out))
print('Saved:', SPLITS_DIR / 'test_v2.csv', '| rows =', len(test_out))
print('train_v2 columns:', len(train_out.columns))
print('test_v2 columns :', len(test_out.columns))
